# Day16 数据摄取

**pandas** 提供 `read_csv`、`read_json`、`read_sql` 读入数据,以及 `to_csv`、`to_json`、`to_sql` 导出数据。每个 cell 独立 import、独立运行、数据硬编码。

In [ ]:
# 导入 pandas,别名为 pd
import pandas as pd

# 构造硬编码学生数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [20, 21, 19, 22],
    "城市": ["北京", "上海", "广州", "深圳"]
})

# 先写出 CSV 文件,模拟真实数据源
df.to_csv("students.csv", index=False, encoding="utf-8")

# 用 read_csv 读回,指定编码
df_read = pd.read_csv("students.csv", encoding="utf-8")
# 打印结果
print(df_read)

**read_csv** 常用参数: `sep` 指定分隔符(默认逗号),`header` 指定列名所在行(默认第 0 行),`index_col` 指定某列作为行索引,`usecols` 只读部分列,`nrows` 只读前 N 行。

In [ ]:
# 导入 pandas
import pandas as pd

# 构造数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, 21, 19],
    "成绩": [88.5, 92.0, 76.5]
})

# 写出以分号分隔的 CSV
df.to_csv("sep.csv", index=False, sep=";", encoding="utf-8")

# 读取时必须指定 sep=";",否则整行会变一列
df_sep = pd.read_csv("sep.csv", sep=";", encoding="utf-8")
print("--- sep=';' ---")
print(df_sep)

# index_col=0:把第 0 列(姓名)设为行索引
df_idx = pd.read_csv("sep.csv", sep=";",
                    index_col=0, encoding="utf-8")
print("\n--- index_col=0 ---")
print(df_idx)

# usecols:只取姓名和成绩两列,节省内存
df_cols = pd.read_csv("sep.csv", sep=";",
                      usecols=["姓名", "成绩"],
                      encoding="utf-8")
print("\n--- usecols ---")
print(df_cols)

# nrows:只读前 2 行,常用于预览大文件
df_n = pd.read_csv("sep.csv", sep=";",
                   nrows=2, encoding="utf-8")
print("\n--- nrows=2 ---")
print(df_n)

**na_values** 参数:把特定字符串(如 "NA"、"-")也视为缺失值 `NaN`。

In [ ]:
# 导入 pandas
import pandas as pd

# 构造含特殊缺失标记的数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, "NA", "-"]
})
# 写出 CSV
df.to_csv("na.csv", index=False, encoding="utf-8")

# 不指定 na_values 时,"NA" 和 "-" 会被当普通字符串
df_default = pd.read_csv("na.csv", encoding="utf-8")
print("--- 不指定 na_values ---")
print(df_default)

# 指定 na_values,把 "NA" 和 "-" 视为缺失值
df_na = pd.read_csv("na.csv", na_values=["NA", "-"],
                   encoding="utf-8")
print("\n--- na_values=['NA','-'] ---")
print(df_na)

**read_json** 可从 JSON 字符串或文件路径读入数据。`orient="records"` 表示列表套字典的格式,最为常用。

In [ ]:
# 导入 pandas
import pandas as pd

# 构造数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, 21, 19]
})

# 将 DataFrame 转为 JSON 字符串(orient="records")
# force_ascii=False 保留中文,不转义为 \uXXXX
json_str = df.to_json(orient="records",
                      force_ascii=False)
print("--- JSON 字符串 ---")
print(json_str)

# 从 JSON 字符串读回 DataFrame
df_back = pd.read_json(json_str)
print("\n--- read_json 读回 ---")
print(df_back)

**read_json 嵌套结构**:当 JSON 是多层嵌套时,可用 `json_normalize` 展平成表格。

In [ ]:
# 导入 pandas
import pandas as pd
# 导入 json 用于处理字符串
import json

# 构造嵌套结构的 Python 字典
data = [
    {
        "姓名": "张三",
        "班级": "一班",
        "成绩": {"语文": 88, "数学": 92}
    },
    {
        "姓名": "李四",
        "班级": "二班",
        "成绩": {"语文": 75, "数学": 85}
    }
]

# 用 json_normalize 展平嵌套字段
df = pd.json_normalize(
    data,
    meta=["姓名", "班级"]
)
print("--- 嵌套 JSON 展平结果 ---")
print(df)

**read_sql** 从 SQLite 数据库读入数据。标准流程:`sqlite3.connect()` 建立连接 → 创建表并插入数据 → `pd.read_sql()` 把查询结果读成 DataFrame → **用 with 上下文管理器自动关闭连接**。

In [ ]:
# 导入 pandas
import pandas as pd
# 导入 sqlite3
import sqlite3

# 使用 with 上下文管理器,自动关闭连接
with sqlite3.connect(":memory:") as conn:
    # 获取游标
    cur = conn.cursor()
    # 创建 students 表
    cur.execute(
        "CREATE TABLE students "
        "(id INTEGER PRIMARY KEY, name TEXT, age INTEGER)"
    )
    # 批量插入数据
    rows = [(1, "张三", 20), (2, "李四", 21),
            (3, "王五", 19), (4, "赵六", 22)]
    cur.executemany(
        "INSERT INTO students VALUES (?, ?, ?)",
        rows
    )
    # 提交事务
    conn.commit()
    # 使用 read_sql 读取全部数据
    df = pd.read_sql("SELECT * FROM students", conn)
    print("--- students 表全部数据 ---")
    print(df)

    # 带条件查询
    df_where = pd.read_sql(
        "SELECT * FROM students WHERE age > 20", conn
    )
    print("\n--- age > 20 的查询结果 ---")
    print(df_where)
# with 块结束,连接自动关闭

**to_csv** 导出 DataFrame 到 CSV 文件。**必须加 `index=False`**,否则会多出一列行索引。可指定 `encoding` 防止中文乱码。

In [ ]:
# 导入 pandas
import pandas as pd

# 构造数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, 21, 19],
    "成绩": [88.5, 92.0, 76.5]
})

# 默认导出:会多出一列行索引(0,1,2)
df.to_csv("with_index.csv", encoding="utf-8")
print("--- 默认导出(含行索引) ---")
print(pd.read_csv("with_index.csv"))

# index=False:去掉行索引,导出更干净
df.to_csv("no_index.csv", index=False, encoding="utf-8")
print("\n--- index=False 导出 ---")
print(pd.read_csv("no_index.csv"))

**to_json** 导出 DataFrame 到 JSON 字符串或文件。`orient="records"` 最常用。`force_ascii=False` 保留中文。

In [ ]:
# 导入 pandas
import pandas as pd

# 构造数据
df = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "年龄": [20, 21]
})

# 默认 force_ascii=True,中文会被转义为 \uXXXX
print("--- force_ascii=True(默认) ---")
print(df.to_json(orient="records"))

# force_ascii=False 保留中文
print("\n--- force_ascii=False ---")
print(df.to_json(orient="records",
                 force_ascii=False))

# 导出到 JSON 文件,加 indent=2 美化格式
df.to_json("output.json", orient="records",
           force_ascii=False, indent=2)
print("\n--- 已写出 output.json ---")

**StringIO** 模拟文件对象。用 `io.StringIO` 把字符串包装成类文件对象,可直接传给 `read_csv`,无需写入磁盘,适合测试。

In [ ]:
# 导入 pandas
import pandas as pd
# 导入 io 模块中的 StringIO
from io import StringIO

# 硬编码模拟 CSV 文件内容
csv_content = "姓名,年龄,城市\n张三,20,北京\n李四,21,上海\n王五,19,广州\n"

# 用 StringIO 把字符串包装成类文件对象
fake_file = StringIO(csv_content)

# 直接传给 read_csv,无需写入磁盘
df = pd.read_csv(fake_file)
print("--- StringIO 模拟 CSV 读取 ---")
print(df)

**to_sql** 导出 DataFrame 到 SQLite 数据库。`if_exists` 参数控制行为:"fail"(已存在则报错)、"replace"(覆盖)、"append"(追加)。

In [ ]:
# 导入 pandas
import pandas as pd
# 导入 sqlite3
import sqlite3

# 构造数据
df = pd.DataFrame({
    "name": ["张三", "李四", "王五"],
    "age": [20, 21, 19]
})

# 连接内存数据库
with sqlite3.connect(":memory:") as conn:
    # if_exists="replace":表已存在则覆盖
    df.to_sql("students", conn,
              if_exists="replace", index=False)

    # 读回验证
    df_back = pd.read_sql(
        "SELECT * FROM students", conn
    )
    print("--- to_sql 导出后读回 ---")
    print(df_back)

    # 追加一行数据(新 DataFrame)
    df_new = pd.DataFrame({
        "name": ["赵六"],
        "age": [22]
    })
    # if_exists="append":追加到已有表
    df_new.to_sql("students", conn,
                  if_exists="append", index=False)

    # 再次读回验证
    df_all = pd.read_sql(
        "SELECT * FROM students", conn
    )
    print("\n--- 追加后全部数据 ---")
    print(df_all)

更多练习:https://pandas.pydata.org/docs/user_guide/io.html